# Activity: Inside a Vision Transformer

A Vision Transformer (ViT) reads an image as a sequence of patches. In this activity you build that pipeline step by step and then look inside a pretrained ViT:

1. Cut an image into 16 × 16 patches.
2. Turn the patches into tokens: projection, CLS token and position embeddings.
3. Assemble one encoder block and a tiny ViT.
4. Classify a photo with a pretrained ViT and look at its attention.

Complete every cell marked `# TODO`. After each one, run the check cell below it: it prints **Correct** when your code works. Part 4 is already written: run it and answer the questions.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from skimage import data, transform

torch.manual_seed(0)
image = transform.resize(data.chelsea()[:, 75:375], (224, 224), anti_aliasing=True)  # a cat, 224 x 224 x 3
plt.imshow(image)
plt.axis("off")
plt.show()

## Part 1. From an image to patches

Write `patchify`, which cuts the image into patches of `p × p` pixels and flattens each one. It returns an array of shape `(number_of_patches, p * p * channels)`, with patches in reading order (left to right, top to bottom).

*Hint:* `reshape` the image into a grid of blocks, reorder the axes with `transpose`, and `reshape` again.

In [ ]:
def patchify(img, p=16):
    H, W, C = img.shape
    # TODO: return an array of shape (H // p * W // p, p * p * C).
    return None

In [ ]:
def check_patchify():
    patches = patchify(image)
    if patches is None:
        print("Not yet: patchify() still returns None.")
        return
    if patches.shape != (196, 768):
        print(f"Check the shape: expected (196, 768), got {patches.shape}.")
    elif np.allclose(patches[15], image[16:32, 16:32].reshape(-1)):
        print("Correct: 196 patches of 768 numbers, in reading order.")
    else:
        print("The shape is right, but the order is not: patch 15 should be row 2, column 2.")


check_patchify()

In [ ]:
patches = patchify(image)
if patches is not None:
    fig, axes = plt.subplots(1, 14, figsize=(14, 1.4))
    for k, ax in enumerate(axes):
        ax.imshow(patches[14 * 6 + k].reshape(16, 16, 3))
        ax.axis("off")
    fig.suptitle("Row 7 of the image as a sequence of 14 patches")
    plt.show()

## Part 2. From patches to tokens

Each flattened patch (768 numbers) goes through one linear layer that turns it into a token of size `dim`. Then the model:

- puts a learned **CLS token** at the start of the sequence, and
- adds a learned **position embedding** to every token, so the model knows where each patch came from.

Complete `forward`. It receives a batch of patches with shape `(batch, 196, 768)` and must return `(batch, 197, dim)`.

*Hint:* `self.cls.expand(B, -1, -1)` repeats the CLS token for every image in the batch, and `torch.cat(..., dim=1)` joins it with the patch tokens.

In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, patch_dim=768, num_patches=196, dim=64):
        super().__init__()
        self.proj = nn.Linear(patch_dim, dim)
        self.cls = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, num_patches + 1, dim) * 0.02)

    def forward(self, x):
        B = x.shape[0]
        # TODO: project the patches, put the CLS token first, then add the position embeddings.
        return None

In [ ]:
def check_embedding():
    torch.manual_seed(0)
    emb = PatchEmbedding()
    x = torch.tensor(patchify(image), dtype=torch.float32).unsqueeze(0) if patchify(image) is not None else torch.randn(1, 196, 768)
    out = emb(x)
    if out is None:
        print("Not yet: forward() still returns None.")
        return
    if out.shape != (1, 197, 64):
        print(f"Check the shape: expected (1, 197, 64), got {tuple(out.shape)}.")
        return
    if not torch.allclose(out[0, 0], emb.cls[0, 0] + emb.pos[0, 0]):
        print("The shape is right, but token 0 should be the CLS token plus its position embedding.")
        return
    if not torch.allclose(out[0, 1], emb.proj(x[0, 0]) + emb.pos[0, 1], atol=1e-6):
        print("Check the order: token 1 should be the first patch, projected, plus position embedding 1.")
        return
    print("Correct: 197 tokens of size 64, CLS first, every token with its position.")


check_embedding()

## Part 3. One encoder block and a tiny ViT

A ViT block applies layer normalization **before** each sub-layer and adds a residual connection **after** it:

$$x \leftarrow x + \mathrm{MSA}(\mathrm{LN}(x)), \qquad x \leftarrow x + \mathrm{MLP}(\mathrm{LN}(x))$$

Complete `forward`. The layers are already created in `__init__`.

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, dim=64, heads=4, mlp_dim=128):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim))

    def forward(self, x):
        # TODO: attention sub-layer with its residual connection.
        #       nn.MultiheadAttention returns (output, weights); use h, _ = self.attn(h, h, h).
        # TODO: MLP sub-layer with its residual connection.
        return None

In [ ]:
def check_block():
    torch.manual_seed(0)
    block = EncoderBlock()
    tokens_in = torch.randn(1, 197, 64)
    out = block(tokens_in)
    if out is None:
        print("Not yet: forward() still returns None.")
        return
    if out.shape != tokens_in.shape:
        print(f"Check the shape: a block must keep it, expected {tuple(tokens_in.shape)}, got {tuple(out.shape)}.")
        return
    # residual check: with the sub-layers switched off, the block must return its input
    for layer in (block.attn.out_proj, block.mlp[2]):
        nn.init.zeros_(layer.weight)
        nn.init.zeros_(layer.bias)
    if torch.allclose(block(tokens_in), tokens_in):
        print("Correct: the block keeps the shape (1, 197, 64) and both residual connections are in place.")
    else:
        print("The shape is right, but a residual connection is missing: add x back after each sub-layer.")


check_block()

Now the pieces fit together. This tiny ViT is already written: patches, tokens, a stack of blocks, and a classification head that reads **only the CLS token**. It is not trained, so its predictions are random; the point is to follow the shapes.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, dim=64, depth=4, num_classes=10):
        super().__init__()
        self.embed = PatchEmbedding(dim=dim)
        self.blocks = nn.Sequential(*[EncoderBlock(dim=dim) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, patches):
        x = self.embed(patches)          # (B, 197, dim)
        x = self.blocks(x)               # (B, 197, dim)
        cls = self.norm(x[:, 0])         # the CLS token only: (B, dim)
        return self.head(cls)            # (B, num_classes)


if patchify(image) is not None and PatchEmbedding()(torch.zeros(1, 196, 768)) is not None and EncoderBlock()(torch.zeros(1, 197, 64)) is not None:
    model = TinyViT()
    x = torch.tensor(patchify(image), dtype=torch.float32).unsqueeze(0)
    print("input patches:", tuple(x.shape))
    print("tokens:       ", tuple(model.embed(x).shape))
    print("logits:       ", tuple(model(x).shape))
    print("parameters:   ", sum(p.numel() for p in model.parameters()))
else:
    print("Complete Parts 1 to 3 first.")

## Part 4. Inside a pretrained ViT

This part is already written. It downloads two pretrained models from Hugging Face (about 430 MB in total; it may take a minute):

- **ViT-Base** (`google/vit-base-patch16-224`), trained to classify ImageNet's 1,000 classes.
- **DINO ViT-S/16** (`facebook/dino-vits16`), trained **without labels**.

Run the cells and look at the results.

In [ ]:
from transformers import ViTForImageClassification, ViTImageProcessor, ViTModel

img = (image * 255).astype(np.uint8)

vit_id = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(vit_id)
vit = ViTForImageClassification.from_pretrained(vit_id).eval()

with torch.no_grad():
    probs = vit(**processor(img, return_tensors="pt")).logits[0].softmax(-1)
top = probs.topk(3)
for p, i in zip(top.values, top.indices):
    print(f"{vit.config.id2label[int(i)]:<30} {float(p):.1%}")
print("parameters:", sum(p.numel() for p in vit.parameters()))

In [ ]:
dino_id = "facebook/dino-vits16"
dino_processor = ViTImageProcessor.from_pretrained(dino_id)
dino = ViTModel.from_pretrained(dino_id, add_pooling_layer=False, attn_implementation="eager").eval()

with torch.no_grad():
    inputs = dino_processor(img, return_tensors="pt", size={"height": 224, "width": 224})
    att = dino(**inputs, output_attentions=True).attentions[-1][0, :, 0, 1:]   # heads x 196: what CLS looks at

heads = att.shape[0]
fig, axes = plt.subplots(1, heads + 1, figsize=(3 * (heads + 1), 3))
axes[0].imshow(image)
axes[0].set_title("image")
for h in range(heads):
    axes[h + 1].imshow(image)
    axes[h + 1].imshow(transform.resize(att[h].reshape(14, 14).numpy(), (224, 224)), cmap="inferno", alpha=0.7)
    axes[h + 1].set_title(f"head {h + 1}")
for ax in axes:
    ax.axis("off")
plt.show()

## Questions

Answer briefly in this cell.

1. A 384 × 384 image with the same patch size: how many patch tokens does it produce? How many entries does the attention matrix of one head have, including the CLS token?
2. In Part 2, what would the model lose if you removed the position embeddings?
3. The classification head reads only the CLS token. How can one token decide the class of the whole image?
4. Look at the DINO maps. Which part of the cat's face does each head focus on? This model never saw a label: why is that surprising?

*Your answers:*